In [1]:
import polars as pl
from Python import config
from Python.pipeline import rolling
from Python import pitcher_rolling, batter_rolling

pitcher_games = pl.read_parquet(config.PITCHER_GAMES_PATH)
batter_games = pl.read_parquet(config.BATTER_GAMES_PATH)
pitcher_games.shape, batter_games.shape

((14124, 137), (146385, 46))

In [2]:
print([n for n in dir(pitcher_rolling) if n.isupper()])
print([n for n in dir(batter_rolling) if n.isupper()])

['DEFAULT_MEAN_COLS', 'DEFAULT_MEAN_WINDOWS', 'DEFAULT_RATE_STATS', 'DEFAULT_RATE_WINDOWS', 'FANGRAPHS_FIP_CONSTANT', '_FIP_COUNTS', '_ORDER', '_PITCH_TYPES']
['DEFAULT_EXTRA_RATE_STATS', 'DEFAULT_FALLBACK_K_RATE', 'DEFAULT_SHRINK_PA', 'DEFAULT_WINDOWS', '_ORDER']


In [3]:
pitcher_rolled_raw = rolling.build_pitcher_rolling(pitcher_games, keep_raw=True)
batter_rolled_raw = rolling.build_batter_rolling(batter_games, keep_raw=True)
pitcher_rolled_raw.shape, batter_rolled_raw.shape

((14124, 358), (146385, 56))

In [4]:
check = (
    pitcher_rolled_raw
    .filter(pl.col("season") == 2025)
    .select(
        pl.col("k_rate_P20").mean().alias("mean_rolling"),
        (pl.col("K").sum() / pl.col("PA").sum()).alias("mean_actual_rate"),
    )
)
check

mean_rolling,mean_actual_rate
f64,f64
0.220197,0.217691


In [5]:
first_starts = (
    pitcher_rolled_raw
    .sort(["pitcher", "game_date"])
    .group_by("pitcher", maintain_order=True)
    .head(1)
    .select("pitcher", "game_date", "season", pl.selectors.matches("_P\\d+"))
)
first_starts.select(pl.selectors.matches("_P\\d+").null_count())

k_rate_P5,k_rate_P10,k_rate_P20,bb_rate_P5,bb_rate_P10,bb_rate_P20,csw_rate_P5,csw_rate_P10,csw_rate_P20,swstr_rate_P5,swstr_rate_P10,swstr_rate_P20,whiff_rate_P5,whiff_rate_P10,whiff_rate_P20,cs_rate_P5,cs_rate_P10,cs_rate_P20,chase_rate_P5,chase_rate_P10,chase_rate_P20,zone_rate_P5,zone_rate_P10,zone_rate_P20,contact_rate_P5,contact_rate_P10,contact_rate_P20,gb_rate_P5,gb_rate_P10,gb_rate_P20,hr_rate_P5,hr_rate_P10,hr_rate_P20,ff_velo_P3,ff_velo_P5,ff_velo_P10,ff_spinrate_P3,…,cu_usage_vL_P10,ch_usage_vR_P3,ch_usage_vR_P5,ch_usage_vR_P10,ch_usage_vL_P3,ch_usage_vL_P5,ch_usage_vL_P10,extension_P3,extension_P5,extension_P10,rel_x_P3,rel_x_P5,rel_x_P10,rel_z_P3,rel_z_P5,rel_z_P10,rel_x_sd_P3,rel_x_sd_P5,rel_x_sd_P10,rel_z_sd_P3,rel_z_sd_P5,rel_z_sd_P10,xBA_P3,xBA_P5,xBA_P10,wOBA_P3,wOBA_P5,wOBA_P10,xwOBA_P3,xwOBA_P5,xwOBA_P10,FIP_P3,xFIP_P3,FIP_P5,xFIP_P5,FIP_P10,xFIP_P10
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,…,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535,535


In [6]:
cutoff_date = pl.date(2025, 7, 1)
truncated_games = pitcher_games.filter(pl.col("game_date") < cutoff_date)

rolled_truncated = rolling.build_pitcher_rolling(truncated_games, keep_raw=True)
rolled_full = pitcher_rolled_raw

compare = (
    rolled_truncated.filter(pl.col("game_date") == cutoff_date - pl.duration(days=1))
    .join(
        rolled_full.select("pitcher", "game_date", pl.col("k_rate_P20").alias("k_rate_P20_full")),
        on=["pitcher", "game_date"],
    )
    .with_columns((pl.col("k_rate_P20") == pl.col("k_rate_P20_full")).alias("matches_full_history"))
)
compare.select("pitcher", "game_date", "k_rate_P20", "k_rate_P20_full", "matches_full_history")

pitcher,game_date,k_rate_P20,k_rate_P20_full,matches_full_history
i64,date,f64,f64,bool
453286,2025-06-30,0.252404,0.252404,true
554430,2025-06-30,0.330579,0.330579,true
571578,2025-06-30,0.195699,0.195699,true
571760,2025-06-30,0.178649,0.178649,true
607074,2025-06-30,0.282378,0.282378,true
…,…,…,…,…
669432,2025-06-30,0.17713,0.17713,true
669923,2025-06-30,0.213823,0.213823,true
676979,2025-06-30,0.320755,0.320755,true


In [7]:
pitcher_rolled_trimmed = rolling.build_pitcher_rolling(pitcher_games, keep_raw=False)
set(pitcher_rolled_raw.columns) - set(pitcher_rolled_trimmed.columns)

{'BB',
 'BIP',
 'Balls',
 'CS',
 'CSW',
 'Chases',
 'Contacts',
 'FB',
 'FIP',
 'GB',
 'HBP',
 'HR',
 'Hits',
 'InZone',
 'OContacts',
 'OutZone',
 'Pitches',
 'Runs',
 'Strikes',
 'Swings',
 'Whiffs',
 'ZContacts',
 'ZSwings',
 'ch_hb',
 'ch_ivb',
 'ch_spinrate',
 'ch_usage_vL',
 'ch_usage_vR',
 'ch_vaa',
 'ch_velo',
 'ch_woba',
 'ch_xwoba',
 'chase_rate',
 'contact_rate',
 'cu_hb',
 'cu_ivb',
 'cu_spinrate',
 'cu_usage_vL',
 'cu_usage_vR',
 'cu_vaa',
 'cu_velo',
 'cu_woba',
 'cu_xwoba',
 'extension',
 'fc_hb',
 'fc_ivb',
 'fc_spinrate',
 'fc_usage_vL',
 'fc_usage_vR',
 'fc_vaa',
 'fc_velo',
 'fc_woba',
 'fc_xwoba',
 'ff_hb',
 'ff_ivb',
 'ff_spinrate',
 'ff_usage_vL',
 'ff_usage_vR',
 'ff_vaa',
 'ff_velo',
 'ff_woba',
 'ff_xwoba',
 'fs_hb',
 'fs_ivb',
 'fs_spinrate',
 'fs_usage_vL',
 'fs_usage_vR',
 'fs_vaa',
 'fs_velo',
 'fs_woba',
 'fs_xwoba',
 'lg_hr_fb_prior',
 'ocontact_rate',
 'rel_x',
 'rel_x_sd',
 'rel_z',
 'rel_z_sd',
 'si_hb',
 'si_ivb',
 'si_spinrate',
 'si_usage_vL',
 'si_